# Dataset Prep — Zip to Train/Valid/Test

Run this whenever you add new medicine strip photos. It takes a zip
of raw images and merges them into your existing `train/images`,
`valid/images`, `test/images` folders with a 70/20/10 split.

**Before running:** upload your new zip file into the same folder
as this notebook (e.g. via Jupyter's Upload button), then set
`ZIP_PATH` below to its filename.

In [ ]:
import os, shutil, zipfile, random

ZIP_PATH = "New folder (2).zip"   # <-- change to your uploaded zip's filename
EXTRACT_DIR = "_dataset_prep_tmp"
PREFIX = "medicinestrip"          # filename prefix for renamed images
SPLIT_RATIOS = (0.7, 0.2, 0.1)    # train, valid, test

PROJECT_ROOT = "."  # assumes train/ valid/ test/ already exist here

In [ ]:
# 1. Unzip
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# Find all images anywhere inside the extracted folder (handles nested folders)
image_paths = []
for root, _, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(root, f))

print(f"Found {len(image_paths)} images in {ZIP_PATH}")

In [ ]:
# 2. Shuffle and split
random.seed(42)
random.shuffle(image_paths)

n = len(image_paths)
n_train = max(1, int(n * SPLIT_RATIOS[0]))
n_valid = max(1, int(n * SPLIT_RATIOS[1]))
n_test = max(1, n - n_train - n_valid)
if n_train + n_valid + n_test > n:
    n_train = n - n_valid - n_test

splits = {
    "train": image_paths[:n_train],
    "valid": image_paths[n_train:n_train + n_valid],
    "test": image_paths[n_train + n_valid:],
}

for name, files in splits.items():
    print(f"{name}: {len(files)} images")

In [ ]:
# 3. Copy into train/valid/test/images, auto-numbered to avoid filename clashes
for split_name, files in splits.items():
    img_dir = os.path.join(PROJECT_ROOT, split_name, "images")
    lbl_dir = os.path.join(PROJECT_ROOT, split_name, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    existing = len(os.listdir(img_dir))
    for i, src in enumerate(files, start=existing + 1):
        ext = os.path.splitext(src)[1].lower()
        dst_name = f"{PREFIX}_{split_name}_{i:03d}{ext}"
        shutil.copy(src, os.path.join(img_dir, dst_name))

shutil.rmtree(EXTRACT_DIR)
print("Done. Images copied into train/valid/test/images/.")
print("NOTE: these new images still need bounding-box labels before training")
print("      (use Roboflow or LabelImg, then drop the .txt files into the")
print("      matching labels/ folder).")

In [ ]:
# 4. Confirm counts after merge
for split_name in ["train", "valid", "test"]:
    img_count = len(os.listdir(os.path.join(PROJECT_ROOT, split_name, "images")))
    lbl_count = len(os.listdir(os.path.join(PROJECT_ROOT, split_name, "labels")))
    print(f"{split_name}: {img_count} images, {lbl_count} labels")